In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
file_path = '/content/drive/My Drive/Project_DataMining/data/raw/Predict Hair Fall.csv'
import pandas as pd
df=pd.read_csv(file_path)

In [ ]:
df.shape

(999, 13)

In [ ]:
df.duplicated().sum()

np.int64(0)

In [ ]:
#vì có các hàng trùng lặp nên xóa các hàng trùng lặp
df.drop_duplicates(inplace=True)

In [ ]:
#Tính tỉ lệ giá trị trống
missing_ratio = df.isnull().mean() * 100
print("Tỷ lệ giá trị trống (%):")
print(missing_ratio)

Tỷ lệ giá trị trống (%):
Id                           0.0
Genetics                     0.0
Hormonal Changes             0.0
Medical Conditions           0.0
Medications & Treatments     0.0
Nutritional Deficiencies     0.0
Stress                       0.0
Age                          0.0
Poor Hair Care Habits        0.0
Environmental Factors        0.0
Smoking                      0.0
Weight Loss                  0.0
Hair Loss                    0.0
dtype: float64


In [ ]:
df.columns = df.columns.str.strip().str.replace(' ', '_')

df.head()

,Id,Genetics,Hormonal_Changes,Medical_Conditions,Medications_&_Treatments,Nutritional_Deficiencies,Stress,Age,Poor_Hair_Care_Habits,Environmental_Factors,Smoking,Weight_Loss,Hair_Loss
0,133992,Yes,No,No Data,No Data,Magnesium deficiency,Moderate,19,Yes,Yes,No,No,0
1,148393,No,No,Eczema,Antibiotics,Magnesium deficiency,High,43,Yes,Yes,No,No,0
2,155074,No,No,Dermatosis,Antifungal Cream,Protein deficiency,Moderate,26,Yes,Yes,No,Yes,0
3,118261,Yes,Yes,Ringworm,Antibiotics,Biotin Deficiency,Moderate,46,Yes,Yes,No,No,0
4,111915,No,No,Psoriasis,Accutane,Iron deficiency,Moderate,30,No,Yes,Yes,No,1


In [ ]:
df = df.drop('Id', axis = 1)

In [ ]:
# Phân loại cột
categorical_columns = ['Medical_Conditions', 'Medications_&_Treatments', 'Nutritional_Deficiencies', 'Stress']
binary_columns = ['Genetics', 'Hormonal_Changes', 'Poor_Hair_Care_Habits', 'Environmental_Factors', 'Smoking', 'Weight_Loss']
numerical_columns = ['Age', 'Hair_Loss']

In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.impute import KNNImputer, SimpleImputer

# Tự lấy cột categorical còn tồn tại trong df
categorical_columns = df.select_dtypes(include=['object']).columns.tolist()

if len(categorical_columns) > 0:
    cat_imputer = SimpleImputer(strategy='most_frequent')
    df[categorical_columns] = cat_imputer.fit_transform(df[categorical_columns])

    ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
    ohe_data = ohe.fit_transform(df[categorical_columns])

    ohe_columns = ohe.get_feature_names_out(categorical_columns)
    ohe_df = pd.DataFrame(ohe_data, columns=ohe_columns, index=df.index)

    df = pd.concat([df, ohe_df], axis=1)
else:
    print("Không có cột categorical để xử lý")

# Chuẩn hóa numerical columns
scaler = StandardScaler()
df[numerical_columns] = scaler.fit_transform(df[numerical_columns])

Không có cột categorical để xử lý


In [ ]:
# Chuẩn bị dữ liệu
X = df.drop(columns=['Hair_Loss'])
y = df['Hair_Loss'].astype(int)  # chắc chắn nhãn là int 0/1

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size=0.3,
                                                    random_state=42,
                                                    stratify=y)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
# Huấn luyện Logistic Regression
log_model = LogisticRegression(max_iter=1000, random_state=42)
log_model.fit(X_train, y_train)
log_pred = log_model.predict(X_test)
print(f"Logistic Regression Accuracy: {accuracy_score(y_test, log_pred):.4f}")

Logistic Regression Accuracy: 0.4700


In [ ]:
from sklearn.svm import SVC
# Huấn luyện SVM
svm_model = SVC(kernel='linear', random_state=42)
svm_model.fit(X_train, y_train)
svm_pred = svm_model.predict(X_test)
print(f"SVM Accuracy: {accuracy_score(y_test, svm_pred):.4f}")

SVM Accuracy: 0.4833


In [ ]:
import joblib
# Đường dẫn lưu file
output_path = '/content/drive/My Drive/Project_DataMining/output/'
processed_data_path = '/content/drive/My Drive/Project_DataMining/data/processed/'

# Lưu mô hình
joblib.dump(log_model, output_path + 'trainFirst_Logistic.pkl')
joblib.dump(svm_model, output_path + 'trainFirst_SVM.pkl')
df.to_csv(processed_data_path + 'data_final.csv', index=False)

# Khi cần load lại mô hình và dữ liệu

model_logistic = joblib.load(output_path + 'trainFirst_Logistic.pkl')
model_svm = joblib.load(output_path + 'trainFirst_SVM.pkl')

df_trained = pd.read_csv(processed_data_path + 'data_final.csv')